In [1]:
# module for building the pyomo model
from pyomo.environ import *
import pyomo.environ as pe

# module for solving the pyomo model
import pyomo.opt as po

In [2]:
model = pe.ConcreteModel()

# Notación

In [3]:
necesidades_dict = {
    "N1": 'Financiación de inversiones en activos fijos',
    "N2": 'Capacidad de pago de deuda financiera',
    "N3": 'Colchón de tesorería insuficiente',
    "N4": 'Generación de liquidez deficiente'
}

acciones_dict = {
    "A1": 'Distribución racional de la deuda',
    "A2": 'Reducir deuda estructural',
    "A3": 'Renegociar calendarios de amortización',
    "A4": 'Identificar y aplicar palancas de Ebitda',
    "A5": 'Mejora de la tesorería',
    "A6": 'Aplaza pago de impuestos',
    "A7": 'Mejora la liquidez operativa',
    "A8": 'Solicita nueva financiación',
    "A9": 'Renegocia vencimientos de la deuda',
    "A10": 'Reduce inversión en stocks'
}

# Conjuntos

In [4]:
# Necesidades (N)
model.n = pe.Set(initialize=['N1','N2','N3','N4'])
# Acciones/Planes (A)
model.a = pe.Set(initialize=['A1','A2','A3','A4','A5','A6','A7','A8','A9','A10'])
# Relaciones válidas (R)
model.r = pe.Set(within=model.n * model.a,
                 initialize=[('N1','A1'),('N2','A2'),('N2','A3'),('N2','A4'),
                             ('N3','A5'),('N3','A6'),
                             ('N4','A7'),('N4','A8'),('N4','A9'),('N4','A10')])

# Parámetros

In [5]:
# Parámetros
# Urgencia de necesidades
u = {'N1':1,'N2':2,'N3':2,'N4':3}
# Importancia de necesidades
imp = {'N1':2,'N2':1,'N3':2,'N4':3}
# Plazo de ejecución
p = {'A1':6,'A2':6,'A3':3,'A4':6,'A5':2,'A6':3,'A7':1,'A8':2,'A9':2,'A10':6}
# Complejidad
c = {'A1':2,'A2':1,'A3':1,'A4':2,'A5':1,'A6':1,'A7':2,'A8':1,'A9':1,'A10':2}
# Condiciones de aplicabilidad (1 = aplicable, 0 = no aplicable)
cond = {'A1':1, 'A2':1, 'A3':1, 'A4':1,
        'A5':1, 'A6':1, 'A7':1, 'A8':1, 'A9':1, 'A10':1}

model.u = pe.Param(model.n, initialize=u)
model.imp = pe.Param(model.n, initialize=imp)
model.p = pe.Param(model.a, initialize=p)
model.c = pe.Param(model.a, initialize=c)
model.cond = pe.Param(model.a, initialize=cond)


# Párametros de Ponderación

In [6]:
alpha = 4   # Peso urgencia
beta  = 3   # Peso importancia
gamma = 0.5 # Penalización plazo
delta = 1   # Penalización complejidad

# Variables

In [7]:
# x_ij: acción j asignada a necesidad i
model.x = pe.Var(model.r, within=pe.Binary)
# y_j: la acción j es seleccionada
model.y = pe.Var(model.a, within=pe.Binary)

# Función de Peso

In [8]:
def weight_rule(model,i,j):
    return alpha*model.u[i] + beta*model.imp[i] - gamma*model.p[j] - delta*model.c[j]
model.w = pe.Param(model.n, model.a,
                   initialize={(i,j):weight_rule(model,i,j) for i in model.n for j in model.a})


# Función Objetivo

In [9]:
def obj_rule(model):
    return sum(model.w[i,j]*model.x[(i,j)] for (i,j) in model.r)
model.obj = pe.Objective(rule=obj_rule, sense=pe.maximize)

# Restricciones

In [10]:
## Cobertura de necesidades
def cobertura_rule(model,i):
    return sum(model.x[(i,j)] for j in model.a if (i,j) in model.r) >= 1
model.cobertura = pe.Constraint(model.n, rule=cobertura_rule)

In [11]:
## Coherencia de selección
def coherencia_rule(model, i, j):
    if (i,j) in model.r:
        return model.x[(i,j)] <= model.y[j]
    return pe.Constraint.Skip
model.coherencia = pe.Constraint(model.n, model.a, rule=coherencia_rule)


In [12]:
## Aplicabilidad
def aplicabilidad_rule(model, j):
    return model.y[j] <= model.cond[j]
model.aplicabilidad = pe.Constraint(model.a, rule=aplicabilidad_rule)


In [13]:
## Minimalidad de acciones seleccionadas
M = 5 # máximo de acciones (ejemplo)
model.minimalidad = pe.Constraint(expr=sum(model.y[j] for j in model.a) <= M)


# Resultado

In [14]:
solver = pe.SolverFactory('gurobi')
result = solver.solve(model, tee=True)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2557253
Academic license 2557253 - for non-commercial use only - registered to 20___@alu.comillas.edu
Read LP format model from file C:\Users\carlo\AppData\Local\Temp\tmpft5zhksg.pyomo.lp
Reading time = 0.01 seconds
x1: 25 rows, 20 columns, 50 nonzeros
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-8550U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Academic license 2557253 - for non-commercial use only - registered to 20___@alu.comillas.edu
Optimize a model with 25 rows, 20 columns and 50 nonzeros
Model fingerprint: 0x21614721
Variable types: 0 continuous, 20 integer (20 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+00, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]
Found heuristic solutio

In [15]:
print('Acciones recomendadas:')
for j in model.a:
    if value(model.y[j]) > 0.5:
        print(acciones_dict[j])
print('\nAsignación necesidad – acción:')
for (i,j) in model.r:
    if value(model.x[(i,j)]) > 0.5:
        print(f'(Necesidad) {necesidades_dict[i]} -> (Acción) {acciones_dict[j]}')
print('\nValor objetivo:',round(value(model.obj),2))

Acciones recomendadas:
Distribución racional de la deuda
Renegociar calendarios de amortización
Mejora de la tesorería
Solicita nueva financiación
Renegocia vencimientos de la deuda

Asignación necesidad – acción:
(Necesidad) Financiación de inversiones en activos fijos -> (Acción) Distribución racional de la deuda
(Necesidad) Capacidad de pago de deuda financiera -> (Acción) Renegociar calendarios de amortización
(Necesidad) Colchón de tesorería insuficiente -> (Acción) Mejora de la tesorería
(Necesidad) Generación de liquidez deficiente -> (Acción) Solicita nueva financiación
(Necesidad) Generación de liquidez deficiente -> (Acción) Renegocia vencimientos de la deuda

Valor objetivo: 63.5
